In [1]:
# Cell 1 — Mount Drive and Install
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch --quiet

import os
os.makedirs(
    '/content/drive/MyDrive/scaling_study',
    exist_ok=True
)

import json

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({'step': 'drive mounted and libraries installed'}, f)
    f.write('\n')

print("Done - Saved to Drive")

Mounted at /content/drive
Done - Saved to Drive


In [2]:
# Cell 2 — Imports
import numpy as np
import pandas as pd
import re
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score)

from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)
import torch
from torch.utils.data import Dataset

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({'step': 'all imports successful'}, f)
    f.write('\n')

print("All imports done - Saved to Drive")

All imports done - Saved to Drive


In [3]:
# Cell 3 — Load Dataset From Drive
df = pd.read_csv(
    '/content/drive/MyDrive/WELFake_Dataset.csv'
)

print("Dataset loaded")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'dataset loaded',
        'rows': len(df),
        'columns': df.columns.tolist()
    }, f)
    f.write('\n')

print("Saved to Drive")

Dataset loaded
Shape: (72134, 4)
Columns: ['Unnamed: 0', 'title', 'text', 'label']
Saved to Drive


In [4]:
# Cell 4 — Clean Data
df = df.dropna()
df = df.drop_duplicates()
df = df.reset_index(drop=True)

df['content'] = (df['title'].astype(str) +
                 ' ' +
                 df['text'].astype(str))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['content'] = df['content'].apply(clean_text)

print("Cleaning done")
print("Final shape:", df.shape)

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'data cleaned',
        'final_rows': len(df)
    }, f)
    f.write('\n')

print("Saved to Drive")

Cleaning done
Final shape: (71537, 5)
Saved to Drive


In [5]:
# Cell 5 — Split Data
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'data split done',
        'train_size': len(X_train),
        'test_size': len(X_test)
    }, f)
    f.write('\n')

print("Saved to Drive")

Train size: 57229
Test size: 14308
Saved to Drive


In [6]:
# Cell 6 — Dataset Class
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({'step': 'dataset class defined'}, f)
    f.write('\n')

print("Dataset class ready - Saved to Drive")

Dataset class ready - Saved to Drive


In [7]:
# Cell 7 — Load Tokenizer
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({'step': 'tokenizer loaded successfully'}, f)
    f.write('\n')

print("Tokenizer loaded - Saved to Drive")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded - Saved to Drive


In [8]:
# Cell 8 — Prepare Data for 5000 Samples
SAMPLE_SIZE = 5000

X_sample = X_train[:SAMPLE_SIZE].tolist()
y_sample = y_train[:SAMPLE_SIZE].tolist()

print(f"Training sample size: {len(X_sample)}")
print(f"Test size: {len(X_test)}")

print("Tokenizing training data...")
train_encodings = distilbert_tokenizer(
    X_sample,
    truncation=True,
    padding=True,
    max_length=64
)

print("Tokenizing test data...")
test_encodings = distilbert_tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

train_dataset = FakeNewsDataset(
    train_encodings,
    y_sample
)
test_dataset = FakeNewsDataset(
    test_encodings,
    y_test.tolist()
)

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'data tokenized and prepared',
        'sample_size': SAMPLE_SIZE,
        'train_tokens': len(train_dataset),
        'test_tokens': len(test_dataset)
    }, f)
    f.write('\n')

print("Data preparation done - Saved to Drive")

Training sample size: 5000
Test size: 14308
Tokenizing training data...
Tokenizing test data...
Data preparation done - Saved to Drive


In [9]:
# Cell 9 — Load Model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({'step': 'distilbert model loaded'}, f)
    f.write('\n')

print("Model loaded - Saved to Drive")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded - Saved to Drive


In [10]:
# Cell 10 — Training Arguments
training_args = TrainingArguments(
    output_dir='./distilbert_5000',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_steps=10000,
    eval_strategy='epoch',
    use_cpu=True,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

with open('/content/drive/MyDrive/scaling_study/checkpoint_5000.json', 'a') as f:
    json.dump({
        'step': 'trainer configured',
        'epochs': 2,
        'batch_size': 2,
        'max_length': 64,
        'hardware': 'CPU only'
    }, f)
    f.write('\n')

print('Trainer ready - Saved to Drive')

Trainer ready - Saved to Drive


In [11]:
# Cell 11 — Train
print("Training started for 5000 samples...")
print("This will take around 1 hour...")

trainer.train()

print("Training complete")

with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'training complete',
        'sample_size': 5000,
        'epochs_done': 2
    }, f)
    f.write('\n')

print("Training completion saved to Drive")
print("Run Cell 12 now immediately")

Training started for 5000 samples...
This will take around 1 hour...


Epoch,Training Loss,Validation Loss
1,0.178380,0.288244
2,0.212384,0.225412


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete
Training completion saved to Drive
Run Cell 12 now immediately


In [12]:
model.save_pretrained(
    '/content/drive/MyDrive/scaling_study/distilbert_5000_model'
)
distilbert_tokenizer.save_pretrained(
    '/content/drive/MyDrive/scaling_study/distilbert_5000_model'
)
print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved


In [13]:
# Cell 12 — Evaluate and Save Final Result
print("Evaluating...")

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(
    predictions.predictions, axis=1
)

accuracy = accuracy_score(y_test, pred_labels)
precision = precision_score(y_test, pred_labels)
recall = recall_score(y_test, pred_labels)
f1 = f1_score(y_test, pred_labels)

result = {
    'sample_size': 5000,
    'accuracy': round(accuracy * 100, 2),
    'precision': round(precision * 100, 2),
    'recall': round(recall * 100, 2),
    'f1': round(f1 * 100, 2)
}

print("\n=============================")
print("RESULTS FOR 5000 SAMPLES")
print("=============================")
print(f"Accuracy  : {result['accuracy']}%")
print(f"Precision : {result['precision']}%")
print(f"Recall    : {result['recall']}%")
print(f"F1 Score  : {result['f1']}%")
print("=============================")

# Save final result separately
with open(
    '/content/drive/MyDrive/scaling_study/result_5000.json',
    'w'
) as f:
    json.dump(result, f)

# Also save to checkpoint
with open(
    '/content/drive/MyDrive/scaling_study/checkpoint_5000.json',
    'a'
) as f:
    json.dump({
        'step': 'FINAL RESULT',
        'sample_size': 5000,
        'accuracy': result['accuracy'],
        'precision': result['precision'],
        'recall': result['recall'],
        'f1': result['f1']
    }, f)
    f.write('\n')

print("\nFINAL RESULT SAVED TO DRIVE")
print("Location: scaling_study/result_5000.json")
print("Safe to close Colab now")

Evaluating...



RESULTS FOR 5000 SAMPLES
Accuracy  : 96.07%
Precision : 95.57%
Recall    : 96.69%
F1 Score  : 96.13%

FINAL RESULT SAVED TO DRIVE
Location: scaling_study/result_5000.json
Safe to close Colab now
